In [ ]:
pip install requests beautifulsoup4

In [ ]:
import requests

url = "https://books.toscrape.com/"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=10
)

print(response.status_code)
print(response.text[:500])

200
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://books.toscrape.com/catalogue/category/books_1/"

all_books = []
for page in range(1, 6):
    if page == 1:
        url = BASE_URL + "index.html"
    else:
        url = BASE_URL + f"page-{page}.html"

    print("Scraping:", url)

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=10
    )

    print("Status:", response.status_code)

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.select("article.product_pod")

    print("Books found:", len(books))

    for book in books:

        # Title
        title = book.h3.a["title"]

        # Price
        price = book.select_one(".price_color").get_text(strip=True)

        # Star rating
        rating_class = book.select_one(".star-rating")["class"]

        star_rating = rating_class[1]

        # Availability
        availability = book.select_one(
            ".availability"
        ).get_text(" ", strip=True)

        # Category
        # All Products pages don't directly display category,
        # so we'll visit the individual book page.
        book_url = book.h3.a["href"]

        # Fix relative URL
        if book_url.startswith("../../"):
            book_url = "https://books.toscrape.com/catalogue/" + book_url[6:]

        book_response = requests.get(
            book_url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=10
        )

        book_soup = BeautifulSoup(
            book_response.text,
            "html.parser"
        )

        # Breadcrumb:
        # Home > Books > Category > Book
        breadcrumb = book_soup.select(
            "ul.breadcrumb li a"
        )

        if len(breadcrumb) >= 3:
            category = breadcrumb[2].get_text(strip=True)
        else:
            category = "Unknown"

        all_books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

df_old = pd.DataFrame(all_books)

print("\n==============================")
print("TOTAL BOOKS:", len(df_old))
print("==============================")

print(df_old.head())

print("\nBooks by category:")
print(df_old["category"].value_counts())

# Save CSV
df_old.to_csv("books_dataset.csv", index=False)

print("\nDataset saved as books_dataset.csv")

Scraping: https://books.toscrape.com/catalogue/category/books_1/index.html
Status: 200
Books found: 20
Scraping: https://books.toscrape.com/catalogue/category/books_1/page-2.html
Status: 200
Books found: 20
Scraping: https://books.toscrape.com/catalogue/category/books_1/page-3.html
Status: 200
Books found: 20
Scraping: https://books.toscrape.com/catalogue/category/books_1/page-4.html
Status: 200
Books found: 20
Scraping: https://books.toscrape.com/catalogue/category/books_1/page-5.html
Status: 200
Books found: 20

TOTAL BOOKS: 100
                                   title    price star_rating availability  \
0                   A Light in the Attic  Â£51.77       Three     In stock   
1                     Tipping the Velvet  Â£53.74         One     In stock   
2                             Soumission  Â£50.10         One     In stock   
3                          Sharp Objects  Â£47.82        Four     In stock   
4  Sapiens: A Brief History of Humankind  Â£54.23        Five     In stoc

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"

# 8 specific categories
categories = {
    "Travel": "travel_2",
    "Mystery": "mystery_3",
    "Historical Fiction": "historical-fiction_4",
    "Classics": "classics_6",
    "Science Fiction": "science-fiction_16",
    "Music": "music_14",
    "Business": "business_13",
    "Poetry": "poetry_23"
}

all_books = []

headers = {
    "User-Agent": "Mozilla/5.0"
}
for category_name, category_url in categories.items():   #Scrape for each category

    print(f"\nScraping category: {category_name}")

    page = 1

    while True:

        if page == 1:
            url = (
                BASE_URL
                + f"catalogue/category/books/{category_url}/index.html"
            )
        else:
            url = (
                BASE_URL
                + f"catalogue/category/books/{category_url}/"
                + f"page-{page}.html"
            )

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        books = soup.select("article.product_pod")

        if not books:
            break

        print(f"Books found: {len(books)}")
        for book in books:

            title = book.h3.a.get("title")
            price = book.select_one(".price_color").get_text(strip=True)
            rating_element = book.select_one(".star-rating")
            star_rating = rating_element["class"][1]
            availability = book.select_one(".availability").get_text(" ", strip=True)
            all_books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category_name
            })

        next_button = soup.select_one(
            "li.next a"
        )

        if next_button:
            page += 1
        else:
            break

df = pd.DataFrame(all_books)
df = df.drop_duplicates(
    subset=["title", "category"]
)

df = df.head(100)

print("\n===================================")
print("TOTAL BOOKS:", len(df))
print("===================================")

print("\nBooks per category:")
print(df["category"].value_counts())
df.to_csv(
    "new_books_dataset.csv",
    index=False
)
print("\nDataset saved as books_dataset.csv")


Scraping category: Travel
Books found: 11

Scraping category: Mystery
Books found: 20
Books found: 12

Scraping category: Historical Fiction
Books found: 20
Books found: 6

Scraping category: Classics
Books found: 19

Scraping category: Science Fiction
Books found: 16

Scraping category: Music
Books found: 13

Scraping category: Business

Scraping category: Poetry
Books found: 19

TOTAL BOOKS: 100

Books per category:
category
Mystery               32
Historical Fiction    26
Classics              19
Science Fiction       12
Travel                11
Name: count, dtype: int64

Dataset saved as books_dataset.csv


In [ ]:
df=pd.read_csv("/content/new_books_dataset.csv")

In [ ]:
df

,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
...,...,...,...,...,...
95,Foundation (Foundation (Publication Order) #1),Â£32.42,One,In stock,Science Fiction
96,The Restaurant at the End of the Universe (Hit...,Â£10.92,One,In stock,Science Fiction
97,Ready Player One,Â£19.07,Four,In stock,Science Fiction
98,"Life, the Universe and Everything (Hitchhiker'...",Â£33.26,Two,In stock,Science Fiction


In [ ]:
import numpy as np
import pandas as pd
# £45.17 -> 45.17
df["price_gbp"] = (df["price"].astype(str).str.replace("Â£", "", regex=False).str.strip())
df["price_gbp"] = pd.to_numeric(df["price_gbp"],errors="coerce")

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df["rating"] = df["star_rating"].map(rating_map)

df["in_stock"] = (df["availability"].astype(str).str.contains("In stock", case=False, na=False))

# Median imputation for price,rating
df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
df["rating"] = df["rating"].fillna(df["rating"].median())
df["rating"] = df["rating"].round().astype(int)

df["in_stock"] = df["in_stock"].fillna(False)

df = df[
    [
        "title",
        "price",
        "price_gbp",
        "star_rating",
        "rating",
        "availability",
        "in_stock",
        "category"
    ]
]

print("\nCleaned dataset:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

df.to_csv(
    "books_dataset_cleaned.csv",
    index=False
)

print("\nCleaned dataset saved successfully!")


Cleaned dataset:
                                               title    price  price_gbp  \
0                            It's Only the Himalayas  Â£45.17      45.17   
1  Full Moon over Noahâs Ark: An Odyssey to Mou...  Â£49.43      49.43   
2  See America: A Celebration of Our National Par...  Â£48.87      48.87   
3  Vagabonding: An Uncommon Guide to the Art of L...  Â£36.94      36.94   
4                               Under the Tuscan Sun  Â£37.33      37.33   

  star_rating  rating availability  in_stock category  
0         Two       2     In stock      True   Travel  
1        Four       4     In stock      True   Travel  
2       Three       3     In stock      True   Travel  
3         Two       2     In stock      True   Travel  
4       Three       3     In stock      True   Travel  

Data types:
title            object
price            object
price_gbp       float64
star_rating      object
rating            int64
availability     object
in_stock           bool
category

In [ ]:
df

,title,price,price_gbp,star_rating,rating,availability,in_stock,category
0,It's Only the Himalayas,Â£45.17,45.17,Two,2,In stock,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,49.43,Four,4,In stock,True,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,48.87,Three,3,In stock,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,36.94,Two,2,In stock,True,Travel
4,Under the Tuscan Sun,Â£37.33,37.33,Three,3,In stock,True,Travel
...,...,...,...,...,...,...,...,...
95,Foundation (Foundation (Publication Order) #1),Â£32.42,32.42,One,1,In stock,True,Science Fiction
96,The Restaurant at the End of the Universe (Hit...,Â£10.92,10.92,One,1,In stock,True,Science Fiction
97,Ready Player One,Â£19.07,19.07,Four,4,In stock,True,Science Fiction
98,"Life, the Universe and Everything (Hitchhiker'...",Â£33.26,33.26,Two,2,In stock,True,Science Fiction


In [ ]:
To_Inr=105.50
df['price_Inr']=df['price_gbp']* To_Inr
df['price_Inr']=df['price_Inr'].round(2)
df[['price_gbp','price_Inr']].head(10)

,price_gbp,price_Inr
0,45.17,4765.44
1,49.43,5214.86
2,48.87,5155.78
3,36.94,3897.17
4,37.33,3938.31
5,44.34,4677.87
6,30.54,3221.97
7,56.88,6000.84
8,23.21,2448.66
9,38.95,4109.23


In [ ]:
import sqlite3
conn = sqlite3.connect("books.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON")

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print("Database and tables created successfully.")

Database and tables created successfully.


In [ ]:
categories = df["category"].dropna().unique()

for category in categories:
    cursor.execute(
        """
        INSERT OR IGNORE INTO categories (category_name)
        VALUES (?)
        """,
        (category,)
    )

for _, row in df.iterrows():

    cursor.execute(
        """
        SELECT category_id
        FROM categories
        WHERE category_name = ?
        """,
        (row["category"],)
    )

    category_id = cursor.fetchone()[0]
    cursor.execute(
        """
        INSERT INTO books
        (title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_Inr"],
            row["rating"],
            int(row["in_stock"]),
            category_id
        )
    )


conn.commit()

print("Data inserted successfully.")

Data inserted successfully.


In [ ]:
categories

array(['Travel', 'Mystery', 'Historical Fiction', 'Classics',
       'Science Fiction'], dtype=object)

In [ ]:
cursor.execute("SELECT COUNT(*) FROM categories")
print("Categories:", cursor.fetchone()[0])

cursor.execute("SELECT COUNT(*) FROM books")
print("Books:", cursor.fetchone()[0])

Categories: 5
Books: 100


In [ ]:
cursor.execute("""
SELECT
    books.title,
    categories.category_name
FROM books
JOIN categories
ON books.category_id = categories.category_id
LIMIT 10
""")

for row in cursor.fetchall():
    print(row)

("It's Only the Himalayas", 'Travel')
('Full Moon over Noahâ\x80\x99s Ark: An Odyssey to Mount Ararat and Beyond', 'Travel')
('See America: A Celebration of Our National Parks & Treasured Sites', 'Travel')
('Vagabonding: An Uncommon Guide to the Art of Long-Term World Travel', 'Travel')
('Under the Tuscan Sun', 'Travel')
('A Summer In Europe', 'Travel')
('The Great Railway Bazaar', 'Travel')
('A Year in Provence (Provence #1)', 'Travel')
('The Road to Little Dribbling: Adventures of an American in Britain (Notes From a Small Island #2)', 'Travel')
('Neither Here nor There: Travels in Europe', 'Travel')


In [ ]:
query1 = """
SELECT title, price_gbp, rating, in_stock
FROM books
WHERE rating = 5
"""

result1 = pd.read_sql(query1, conn)

print(result1)

                                                title  price_gbp  rating  \
0                  1,000 Places to See Before You Die      26.08       5   
1              A Time of Torment (Charlie Parker #14)      48.35       5   
2   What Happened on Beale Street (Secrets of the ...      25.37       5   
3   The Bachelor Girl's Guide to Murder (Herringfo...      52.30       5   
4                   The Silkworm (Cormoran Strike #2)      23.05       5   
5                                   The Girl You Lost      12.29       5   
6             A Flight of Arrows (The Pathfinders #2)      55.53       5   
7                                        Mrs. Houdini      30.25       5   
8                               The Passion of Dolssa      28.32       5   
9                              Voyager (Outlander #3)      21.07       5   
10                                       The Red Tent      35.66       5   
11                             Between Shades of Gray      20.79       5   
12          

In [ ]:
query2 = """
SELECT title, price_gbp, price_inr
FROM books ORDER BY price_gbp DESC LIMIT 10
"""
result2 = pd.read_sql(query2, conn)

print("Query 2:")
print(query2)

print("Output:")
print(result2)

Query 2:

SELECT title, price_gbp, price_inr
FROM books ORDER BY price_gbp DESC LIMIT 10

Output:
                                               title  price_gbp  price_inr
0                      Boar Island (Anna Pigeon #19)      59.48    6275.14
1                                            Candide      58.63    6185.46
2  The No. 1 Ladies' Detective Agency (No. 1 Ladi...      57.70    6087.35
3                                        Animal Farm      57.22    6036.71
4                   A Year in Provence (Provence #1)      56.88    6000.84
5                                The Past Never Ends      56.50    5960.75
6                   The Last Painting of Sara de Vos      55.55    5860.52
7            A Flight of Arrows (The Pathfinders #2)      55.53    5858.42
8  Alice in Wonderland (Alice's Adventures in Won...      55.53    5858.42
9                                     Dune (Dune #1)      54.86    5787.73


In [ ]:
query3 = """
SELECT DISTINCT category_name
FROM categories
ORDER BY category_name
"""

result3 = pd.read_sql(query3, conn)
print(result3)

        category_name
0            Classics
1  Historical Fiction
2             Mystery
3     Science Fiction
4              Travel


In [ ]:
query4="""
Select title from books where rating between 2 and 4 order by rating
"""
res4=pd.read_sql(query4,conn)
print(res4)

                                                title
0                             It's Only the Himalayas
1   Vagabonding: An Uncommon Guide to the Art of L...
2                                  A Summer In Europe
3                      The Last Mile (Amos Decker #2)
4             A Study in Scarlet (Sherlock Holmes #1)
..                                                ...
57  The Complete Stories and Poems (The Works of E...
58                          The Story of Hong Gildong
59  William Shakespeare's Star Wars: Verily, A New...
60                                              Arena
61                                   Ready Player One

[62 rows x 1 columns]


In [ ]:
query5 = """
SELECT
    books.title,
    categories.category_name,
    books.rating,
    books.price_gbp,
    books.price_inr
FROM books
JOIN categories
    ON books.category_id = categories.category_id
ORDER BY books.rating DESC, books.price_gbp DESC
LIMIT 10
"""

res5 = pd.read_sql(query5, conn)
print(res5)

                                               title       category_name  \
0            A Flight of Arrows (The Pathfinders #2)  Historical Fiction   
1  The Bachelor Girl's Guide to Murder (Herringfo...             Mystery   
2             A Time of Torment (Charlie Parker #14)             Mystery   
3                                While You Were Mine  Historical Fiction   
4                                               Join     Science Fiction   
5                                       The Red Tent  Historical Fiction   
6                                       Mrs. Houdini  Historical Fiction   
7                              The Passion of Dolssa  Historical Fiction   
8                 1,000 Places to See Before You Die              Travel   
9  What Happened on Beale Street (Secrets of the ...             Mystery   

   rating  price_gbp  price_inr  
0       5      55.53    5858.42  
1       5      52.30    5517.65  
2       5      48.35    5100.92  
3       5      41.32    435

In [ ]:
print("Query1 :",query1)
print("Result1 :",result1)
print("="*100)
print("Query2 :",query2)
print("Result2:",result2)

Query1 : 
SELECT title, price_gbp, rating, in_stock
FROM books
WHERE rating = 5

Result1 :                                                 title  price_gbp  rating  \
0                  1,000 Places to See Before You Die      26.08       5   
1              A Time of Torment (Charlie Parker #14)      48.35       5   
2   What Happened on Beale Street (Secrets of the ...      25.37       5   
3   The Bachelor Girl's Guide to Murder (Herringfo...      52.30       5   
4                   The Silkworm (Cormoran Strike #2)      23.05       5   
5                                   The Girl You Lost      12.29       5   
6             A Flight of Arrows (The Pathfinders #2)      55.53       5   
7                                        Mrs. Houdini      30.25       5   
8                               The Passion of Dolssa      28.32       5   
9                              Voyager (Outlander #3)      21.07       5   
10                                       The Red Tent      35.66       5 

In [ ]:
books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

In [ ]:
merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

In [ ]:
merged_df

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,1,It's Only the Himalayas,45.17,4765.44,2,1,1,Travel
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1,1,Travel
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1,1,Travel
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,1,1,Travel
4,5,Under the Tuscan Sun,37.33,3938.31,3,1,1,Travel
...,...,...,...,...,...,...,...,...
95,96,Foundation (Foundation (Publication Order) #1),32.42,3420.31,1,1,5,Science Fiction
96,97,The Restaurant at the End of the Universe (Hit...,10.92,1152.06,1,1,5,Science Fiction
97,98,Ready Player One,19.07,2011.88,4,1,5,Science Fiction
98,99,"Life, the Universe and Everything (Hitchhiker'...",33.26,3508.93,2,1,5,Science Fiction


In [ ]:
print(books_df.shape)
print(categories_df.shape)
print(merged_df.shape)

(100, 7)
(5, 2)
(100, 8)


In [ ]:
merge_result = merged_df[
    [
        "title",
        "category_name",
        "rating",
        "price_gbp",
        "price_inr"
    ]
].sort_values(
    by=["rating", "price_gbp"],
    ascending=[False, False]
).head(10)

print("JOIN result using pandas merge:")
print(merge_result)

JOIN result using pandas merge:
                                                title       category_name  \
45            A Flight of Arrows (The Pathfinders #2)  Historical Fiction   
29  The Bachelor Girl's Guide to Murder (Herringfo...             Mystery   
19             A Time of Torment (Charlie Parker #14)             Mystery   
64                                While You Were Mine  Historical Fiction   
89                                               Join     Science Fiction   
59                                       The Red Tent  Historical Fiction   
47                                       Mrs. Houdini  Historical Fiction   
56                              The Passion of Dolssa  Historical Fiction   
10                 1,000 Places to See Before You Die              Travel   
28  What Happened on Beale Street (Secrets of the ...             Mystery   

    rating  price_gbp  price_inr  
45       5      55.53    5858.42  
29       5      52.30    5517.65  
19       5     

In [ ]:
res5

,title,category_name,rating,price_gbp,price_inr
0,A Flight of Arrows (The Pathfinders #2),Historical Fiction,5,55.53,5858.42
1,The Bachelor Girl's Guide to Murder (Herringfo...,Mystery,5,52.30,5517.65
2,A Time of Torment (Charlie Parker #14),Mystery,5,48.35,5100.92
3,While You Were Mine,Historical Fiction,5,41.32,4359.26
4,Join,Science Fiction,5,35.67,3763.19
5,The Red Tent,Historical Fiction,5,35.66,3762.13
6,Mrs. Houdini,Historical Fiction,5,30.25,3191.38
7,The Passion of Dolssa,Historical Fiction,5,28.32,2987.76
8,"1,000 Places to See Before You Die",Travel,5,26.08,2751.44
9,What Happened on Beale Street (Secrets of the ...,Mystery,5,25.37,2676.54


In [ ]:
sql_result = res5.reset_index(drop=True)
pandas_result = merge_result.reset_index(drop=True)

print(
    "Both are same??",
    sql_result.equals(pandas_result)
)

Both are same?? True
